In [ ]:
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langchain_core.messages import RemoveMessage,HumanMessage,BaseMessage
from langchain_groq import ChatGroq
from typing import TypedDict,Annotated,List
from dotenv import load_dotenv
import os
from langgraph.checkpoint.memory import InMemorySaver


In [ ]:
load_dotenv(dotenv_path=".env", override=True)
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
model = ChatGroq(
    model="llama-3.3-70b-versatile",  
    api_key=GROQ_API_KEY  
)

In [ ]:
class chatstate(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]
    summary:str

In [ ]:
def summarizer(state:chatstate):
    summary=state['summary']
    messages=state['messages']
    if summary:
        prompt = (
            f"Existing summary:\n{summary}\n\n"
            "Extend the summary using the new conversation above."
        )
    else:
        prompt = "Summarize the conversation above."
        messages_for_summary = state["messages"] + [
        HumanMessage(content=prompt)
    ]
    response=model.invoke(messages_for_summary)
    messages_to_delete = state["messages"][:-2]

    return {
        "summary": response.content,
        "messages": [RemoveMessage(id=m.id) for m in messages_to_delete],
    }

In [ ]:
def chat_node(state: ChatState):
    messages = []

    if state["summary"]:
        messages.append({
            "role": "system",
            "content": f"Conversation summary:\n{state['summary']}"
        })

    messages.extend(state["messages"])

    print(messages)

    response = model.invoke(messages)
    return {"messages": [response]}